# 章节练习

本练习覆盖 **基础概念入门** 章节的核心知识点：GE 的全栈定位与两大职责、编译四阶段与模型下沉调度，以及 AscendIR 构图基础。

## 一、判断题

1. （判断题）GE 是昇腾平台计算图编译和运行的控制中心，提供了图构建、图编译优化及图执行控制等功能。

2. （判断题）AscendIR 是 GE 编译流程使用的核心 IR，用图结构表达模型的计算逻辑与数据依赖关系。

3. （判断题）GE 编译四阶段中，图准备阶段会完成 InferShapeAndType / InferFormat 推导以及常量折叠等操作。

4. （判断题）模型下沉调度场景下，静态图在执行阶段每个算子 Task 都需要由 Host 逐个下发到 Device。

5. （判断题）AscendIR 中，控制边（Control Edge）通过 DataAnchor 进行连接，用于传递 Tensor 数据。

## 二、单选题

6. （单选题）GE 的两大核心职责是什么？
    A. 模型训练和模型评估
    B. 图编译和图执行
    C. 数据预处理和数据增强
    D. 内存管理和网络通信

7. （单选题）AscendIR 的核心对象模型是一条自顶向下、层层持有的链路，下列哪一项正确？
    A. ComputeGraph → Node → Edge → Tensor
    B. ComputeGraph → Node → OpDesc → GeTensorDesc
    C. ComputeGraph → OpDesc → Node → Anchor
    D. Graph → Tensor → Node → OpDesc

8. （单选题）以下关于 GE 编译四阶段的顺序，哪个是正确的？
    A. 图优化 → 图准备 → 图拆分 → 图编译
    B. 图准备 → 图拆分 → 图优化 → 图编译
    C. 图拆分 → 图准备 → 图优化 → 图编译
    D. 图准备 → 图优化 → 图拆分 → 图编译

9. （单选题）GE 编译的图拆分阶段主要完成什么工作？
    A. 将整图按执行引擎（如 AI Core / AI CPU）拆分为多个子图
    B. 推导每个 Tensor 的 shape 和 format
    C. 执行算子融合与 UB 融合
    D. 分配内存与 stream 并生成 .om 文件

10. （单选题）以下关于控制边的描述，哪个是正确的？
    A. 控制边用于传递 Tensor 数据
    B. 控制边表示纯依赖关系，无数据传递，仅约束执行顺序
    C. 控制边和数据边的功能完全相同
    D. 控制边只能用于输入节点和输出节点之间

## 三、多选题

11. （多选题）以下关于模型下沉调度的描述，哪些是正确的？
    A. 静态图在加载阶段将整图下发到 Device
    B. 静态图执行阶段 Host 仅下发一个模型 Task 即可触发整图执行
    C. 动态 Shape 图通常走 Host 调度，由 Host 侧逐步下发 Task
    D. 模型下沉调度可以减少 Host 与 Device 之间的交互开销

12. （多选题）以下关于静态 Shape 图、动态分档和动态 Shape 图的描述，哪些是正确的？
    A. 静态 Shape 图的所有 Tensor shape 在编译期完全确定
    B. 动态分档会设置多个 shape 档位，每个档位按相对确定的静态 shape 方式编译 / 执行
    C. 静态 Shape 图的运行时开销通常低于动态 Shape 图
    D. GE 不支持动态 Shape 图的编译和执行

**执行以下代码获取答案。**


In [ ]:
!cat answer/01.04_answer.txt

## 四、综合编程实践

<div align="left">
<blockquote style="margin-left: 0; margin-right: auto;">
<p>本题为本章的<strong>综合编程实践</strong>（C++），用于检验你对 AscendIR 对象模型、数据边与静态/动态 Shape 的整体掌握。纯 C++ 实现，用 <code>g++</code> 即可编译，<strong>无需 NPU</strong>。</p>
</blockquote>
</div>

**任务**：用 C++ 实现一个简化版 AscendIR 计算图模型，补全 3 处 `TODO`：

1. `Tensor::IsDynamic()`：`shape` 任一维为 `-1` 时判定为动态 Shape。
2. `Graph::TopologicalOrder()`：按**数据边**依赖（节点 `inputs` 引用上游节点的 `output` 名）做拓扑排序，存在环时抛异常。
3. `BuildAddGraph()`：构造 `Data x`、`Data y` → `Add z` → `NetOutput` 的最小图。

**输入/预期**：拓扑序应为 `x, y, add, net_output`；默认静态，将 `x` 的 batch 维改为 `-1` 后应判定为动态图。

运行下面的单元，把**待填写骨架**写入 `Sources/practice/ascendir_mini.cpp`。

In [ ]:
!mkdir -p Sources/practice

In [ ]:
%%writefile Sources/practice/ascendir_mini.cpp
// 1.4 综合编程实践（待填写）：简化版 AscendIR 计算图模型（纯 C++，无需 NPU）
// 补全 3 处 TODO，编译运行：
//   g++ -std=c++17 Sources/practice/ascendir_mini.cpp -o Sources/practice/ascendir_mini
#include <cstdint>
#include <iostream>
#include <map>
#include <queue>
#include <stdexcept>
#include <string>
#include <utility>
#include <vector>

struct Tensor {
  std::string name;
  std::vector<int64_t> shape;
  std::string dtype = "float32";

  bool IsDynamic() const {
    // TODO-1: shape 任一维为 -1 即为动态 Shape，返回 true/false
    return false;
  }
};

struct Node {
  std::string name;
  std::string op_type;
  std::vector<std::string> inputs;  // 依赖的上游输出张量名
  bool has_output = false;
  Tensor output;
};

class Graph {
 public:
  explicit Graph(std::string name) : name_(std::move(name)) {}

  Graph& Add(const Node& node) {
    nodes_.push_back(node);
    if (node.has_output) {
      by_output_[node.output.name] = nodes_.size() - 1;
    }
    return *this;
  }

  std::vector<Node> TopologicalOrder() const {
    // TODO-2: 按数据边依赖(节点 inputs 引用上游 output 名)做 Kahn 拓扑排序，
    //         返回 Node 列表；存在环则 throw std::runtime_error
    return nodes_;
  }

  bool IsDynamicGraph() const {
    for (const auto& n : nodes_) {
      if (n.has_output && n.output.IsDynamic()) {
        return true;
      }
    }
    return false;
  }

  std::vector<Node>& nodes() { return nodes_; }
  const std::string& name() const { return name_; }

 private:
  std::string name_;
  std::vector<Node> nodes_;
  std::map<std::string, size_t> by_output_;
};

Graph BuildAddGraph() {
  // TODO-3: 构造 Data x, Data y -> Add z -> NetOutput 的最小图并返回
  return Graph("add_graph");
}

int main() {
  Graph g = BuildAddGraph();
  std::cout << "图: " << g.name() << ", 节点数: " << g.nodes().size() << "\n";
  std::cout << "拓扑序:\n";
  int i = 1;
  for (const auto& n : g.TopologicalOrder()) {
    std::cout << "  " << i++ << ". " << n.name << " (" << n.op_type << ")\n";
  }
  std::cout << "是否动态 Shape 图: " << std::boolalpha << g.IsDynamicGraph() << "\n";

  if (!g.nodes().empty()) {
    g.nodes()[0].output.shape.assign({-1, 3, 224, 224});
  }
  const bool ok = g.nodes().size() == 4 && g.IsDynamicGraph();
  std::cout << (ok ? "\n[OK] 综合实践通过。" : "\n[FAIL] 请补全 TODO 后重试。") << std::endl;
  return ok ? 0 : 1;
}


补全 `Sources/practice/ascendir_mini.cpp` 的 3 处 `TODO` 后，运行下方单元编译并执行（未补全时输出 `[FAIL]`，属正常现象）：

In [ ]:
# 补全骨架后运行（未补全输出 [FAIL]，属正常）
!g++ -std=c++17 Sources/practice/ascendir_mini.cpp -o Sources/practice/ascendir_mini && ./Sources/practice/ascendir_mini

**参考答案**（建议先独立完成再对照；答案位于 `answer/01.04_practice_answer.cpp`）：

In [ ]:
!cat answer/01.04_practice_answer.cpp

In [ ]:
# 编译运行参考答案查看预期输出
!g++ -std=c++17 answer/01.04_practice_answer.cpp -o /tmp/ascendir_ans && /tmp/ascendir_ans